In [2]:
#Necessary Dependencies 
import sys
sys.path.append("..")
import importlib
import tensorflow as tf
import DataModules.DataAugmentation
importlib.reload(DataModules.DataAugmentation)

from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Flatten, Dropout
from keras.applications.densenet import DenseNet201
from keras.applications.densenet import preprocess_input
from keras.applications.inception_v3 import InceptionV3
from keras.applications.inception_v3 import preprocess_input
from DataModules.DataAugmentation import get_datasets

In [ ]:
train_dataset1, test_dataset1, train_dataset2, test_dataset2, train_dataset3, test_dataset3, train_dataset4, test_dataset4, data_augmentation = get_datasets()

In [ ]:
#Combining the datasets to train the model

combined_train = train_dataset1.concatenate(train_dataset2)
combined_train = combined_train.concatenate(train_dataset3)
combined_train = combined_train.concatenate(train_dataset4)

combined_test = test_dataset1.concatenate(test_dataset2)
combined_test = combined_test.concatenate(test_dataset3)
combined_test = combined_test.concatenate(test_dataset4)

In [ ]:
#The DenseNet201 Model
IMAGE_SIZE = (224, 224)

densenet_ens = DenseNet201(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in densenet_ens.layers:
    layer.trainable = False

x1 = densenet_ens.output
x1 = data_augmentation(x1)
x1 = GlobalAveragePooling2D()(x1)
x1 = Dense(10, activation='relu')(x1)
x1 = Dropout(.5)(x1)
x1 = Flatten()(x1)
densenet_prediction1 = Dense(1, activation='sigmoid')(x1)

# create a model object
densenet_ens = Model(inputs=densenet_rnfl_dm.inputs, outputs=densenet_prediction1)

In [ ]:
#The GoogleNet Model

googlenet_ens = InceptionV3(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in googlenet_ens.layers:
    layer.trainable = False

x2 = googlenet_ens.output
x2 = data_augmentation()(x2)
x2 = GlobalAveragePooling2D()(x2)
x2 = Dense(10, activation='relu')(x2)
x2 = Dropout(.5)(x2)
x2 = Flatten()(x2)
googlenet_prediction2 = Dense(1, activation='sigmoid')(x2)

# create a model object
googlenet_ens = Model(inputs=googlenet_rnfl_tm.inputs, outputs=googlenet_prediction2)

In [ ]:
#The NASNet Model

nasnet_ens = NASNetLarge(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in nasnet_ens.layers:
    layer.trainable = False

x3 = nasnet_ens.output
x3 = data_augmentation()(x3)
x3 = GlobalAveragePooling2D()(x3)
x3 = Dense(10, activation='relu')(x3)
x3 = Dropout(.5)(x3)
x3 = Flatten()(x3)
nasnet_prediction3 = Dense(1, activation='sigmoid')(x3)

# create a model object
nasnet_ens = Model(inputs=nasnet_gcipl_dm.inputs, outputs=nasnet_prediction3)

In [ ]:
#The ResNet Model

resnet_ens = ResNet152V2(input_shape=IMAGE_SIZE + (3,), weights='imagenet', include_top=False)

for layer in resnet_ens.layers:
    layer.trainable = False

x4 = resnet_ens.output
x4 = data_augmentation()(x4)
x4 = GlobalAveragePooling2D()(x4)
x4 = Dense(10, activation='relu')(x4)
x4 = Dropout(.5)(x4)
x4 = Flatten()(x4)
resnet_prediction4 = Dense(1, activation='sigmoid')(x4)

# create a model object
resnet_ens = Model(inputs=resnet_gcipl_tm.inputs, outputs=resnet_prediction4)

In [ ]:
histories = {}

models = {
    "DenseNet201": densenet_ens,
    "GoogleNet": googlenet_ens,
    "NASNet": nasnet_ens,
    "ResNet": resnet_ens
}

for name, model in models.items():
    tf.keras.backend.clear_session()
    print(f"Training {name}...")

    model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
    )
    
    histories[name] = model.fit(
        train_combined,
        validation_data=test_combined,
        epochs=150
    )